# HW3 — Problem 4: Optimizing Waste Management

In [1]:
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

env = gp.Env(empty=True)
env.setParam("OutputFlag", 0)
env.start()

def load(name):
    return pd.read_csv(f"HW3_data/{name}.csv", header=None).to_numpy()

def dist(A, B):
    return np.linalg.norm(A[:, None, :] - B[None, :, :], axis=2)

centers, landfills, stations, q = load("centers"), load("landfills"), load("stations"), load("q").ravel()
centers2, landfills2, stations2, q2 = load("centers2"), load("landfills2"), load("stations2"), load("q2").ravel()

print("Region 1:", centers.shape, landfills.shape, stations.shape, q.shape, f"total waste = {q.sum():,.1f} tons/day")
print("Region 2:", centers2.shape, landfills2.shape, stations2.shape, q2.shape, f"total waste = {q2.sum():,.1f} tons/day")

Region 1: (50, 2) (15, 2) (50, 2) (50,) total waste = 48,803.2 tons/day
Region 2: (40, 2) (15, 2) (50, 2) (40,) total waste = 78,624.9 tons/day


## Part A. Building landfills

### (a) Formulation

Let $I$ be the set of 50 centers and $J$ the set of 15 candidate landfills.

**Decision variables**
- $x_j \in \{0,1\}$: 1 if landfill $j$ is built
- $y_{ij} \in [0,1]$: fraction of the waste from center $i$ sent to landfill $j$

**Model**
$$
\begin{aligned}
\min_{x,\,y}\quad & \sum_{i \in I}\sum_{j \in J} q_i\, d_{ij}\, y_{ij} \\
\text{s.t.}\quad & \sum_{j \in J} y_{ij} = 1 && \forall i \in I \\
& y_{ij} \le x_j && \forall i \in I,\ j \in J \\
& \sum_{j \in J} x_j = 5 \\
& x_j \in \{0,1\},\quad 0 \le y_{ij} \le 1 && \forall i \in I,\ j \in J
\end{aligned}
$$

The first constraint ensures that all waste from each center is disposed of at a landfill. The second constraint only allows waste to be sent to a landfill that is built, and the third requires exactly 5 landfills to be built. In the objective, $q_i y_{ij}$ is the amount of waste sent from center $i$ to landfill $j$, and multiplying by $d_{ij}$ gives the corresponding mile-tons traveled. Therefore, the objective minimizes the total weighted transportation distance. Since the landfills have no capacity constraints, each center will send all of its waste to the nearest open landfill.

### (b) Implementation

In [2]:
def solve_landfills(C, L, q, p):
    d = dist(C, L)
    nC, nL = d.shape

    m = gp.Model(env=env)
    x = m.addMVar(nL, vtype=GRB.BINARY)
    y = m.addMVar((nC, nL), lb=0, ub=1)

    m.addConstr(y.sum(axis=1) == 1)
    m.addConstr(y <= x)
    m.addConstr(x.sum() == p)
    m.setObjective((q[:, None] * d * y).sum(), GRB.MINIMIZE)

    m.optimize()
    assert m.Status == GRB.OPTIMAL
    return m.ObjVal, x.X.round(), y.X

objA, xA, yA = solve_landfills(centers, landfills, q, 5)

assert xA.sum() == 5
assert np.allclose(yA.sum(axis=1), 1)
assert np.all(yA[:, xA == 0] < 1e-6)

print("Landfills built:", np.flatnonzero(xA) + 1)
print(f"Total distance traveled: {objA:,.1f} mile-tons/day")

Landfills built: [ 3  4  7 13 14]
Total distance traveled: 840,487.5 mile-tons/day


**Results.** Build landfills **3, 4, 7, 13, 14**. The total distance traveled by the waste is **840,487.5 mile-tons per day**. Since there are no landfill capacities, each center just sends its waste to the closest of these five.

## Part B. Building landfills and transfer stations

### (c) Formulation

Let $K$ be the set of 50 candidate transfer stations. Distances: $d_{ij}$ (center → landfill), $e_{ik}$ (center → station), $f_{kj}$ (station → landfill). I assume compaction reduces volume but not weight, so the tons that enter a station are the tons that leave it.

**Decision variables**
- $x_j \in \{0,1\}$: 1 if landfill $j$ is built
- $s_k \in \{0,1\}$: 1 if transfer station $k$ is built
- $u_{ij} \ge 0$: tons/day of uncompacted waste sent directly from center $i$ to landfill $j$
- $v_{ik} \ge 0$: tons/day of uncompacted waste sent from center $i$ to station $k$
- $w_{kj} \ge 0$: tons/day of compacted waste sent from station $k$ to landfill $j$

**Model**
$$
\begin{aligned}
\min\quad & \underbrace{\sum_{i,j} 1\cdot d_{ij}\,u_{ij} + \sum_{i,k} 1\cdot e_{ik}\,v_{ik} + \sum_{k,j} 0.5\, f_{kj}\,w_{kj}}_{\text{transportation}} \;+\; \underbrace{10{,}000 \sum_{k} s_k}_{\text{stations}} \\
\text{s.t.}\quad & \sum_{j \in J} u_{ij} + \sum_{k \in K} v_{ik} = q_i && \forall i \in I \\
& \sum_{i \in I} v_{ik} = \sum_{j \in J} w_{kj} && \forall k \in K \\
& \sum_{i \in I} v_{ik} \le 2000\, s_k && \forall k \in K \\
& u_{ij} \le q_i\, x_j && \forall i \in I,\ j \in J \\
& w_{kj} \le 2000\, x_j && \forall k \in K,\ j \in J \\
& \sum_{j \in J} x_j = 5 \\
& x_j, s_k \in \{0,1\},\quad u, v, w \ge 0
\end{aligned}
$$


The first constraint ensures that all $q_i$ tons of waste generated at each center are either sent directly to a landfill or sent to a transfer station. The second constraint enforces flow conservation at each transfer station, so the amount of waste entering a station must equal the amount leaving for landfills. The third constraint limits each transfer station to 2,000 tons per day and prevents waste from entering a station unless it is built. The next two constraints ensure that waste can only be sent to landfills that are built, and the final constraint requires exactly five landfills to be built.

The objective minimizes total daily cost. Waste sent directly from a center to a landfill and waste sent from a center to a transfer station are uncompacted and cost 1 dollar per mile-ton. After passing through a transfer station, the waste is compacted and costs 0.50 dollar per mile-ton to transport to a landfill. Each transfer station that is built also adds \$10,000 per day to the objective.

The total distance traveled by the waste is

$$
\sum_{i,j} d_{ij}u_{ij}
+
\sum_{i,k} e_{ik}v_{ik}
+
\sum_{k,j} f_{kj}w_{kj}.
$$

This measures the total mile-tons traveled along all legs of the network. 

### (d) Implementation

In [3]:
CAP, STATION_COST, C_UNC, C_COMP = 2000, 10_000, 1.0, 0.5

def solve_network(C, L, S, q, p):
    d, e, f = dist(C, L), dist(C, S), dist(S, L)
    nC, nL, nS = len(C), len(L), len(S)

    m = gp.Model(env=env)
    x = m.addMVar(nL, vtype=GRB.BINARY)
    s = m.addMVar(nS, vtype=GRB.BINARY)
    u = m.addMVar((nC, nL))
    v = m.addMVar((nC, nS))
    w = m.addMVar((nS, nL))

    m.addConstr(u.sum(axis=1) + v.sum(axis=1) == q)
    m.addConstr(v.sum(axis=0) == w.sum(axis=1))
    m.addConstr(v.sum(axis=0) <= CAP * s)
    m.addConstr(u <= q[:, None] * x)
    m.addConstr(w <= CAP * x)
    m.addConstr(x.sum() == p)

    m.setObjective(C_UNC * (d * u).sum() + C_UNC * (e * v).sum() + C_COMP * (f * w).sum()
                   + STATION_COST * s.sum(), GRB.MINIMIZE)
    m.optimize()
    assert m.Status == GRB.OPTIMAL

    r = dict(obj=m.ObjVal, x=x.X.round(), s=s.X.round(), u=u.X, v=v.X, w=w.X)
    r["miles"] = (d * r["u"]).sum() + (e * r["v"]).sum() + (f * r["w"]).sum()
    r["transport"] = C_UNC * ((d * r["u"]).sum() + (e * r["v"]).sum()) + C_COMP * (f * r["w"]).sum()
    r["stations_cost"] = STATION_COST * r["s"].sum()
    return r

def check(r, q, p, tol=1e-6):
    u, v, w, x, s = r["u"], r["v"], r["w"], r["x"], r["s"]
    inflow = v.sum(axis=0)
    assert x.sum() == p
    assert np.allclose(u.sum(axis=1) + v.sum(axis=1), q)    # all waste collected
    assert np.allclose(inflow, w.sum(axis=1))               # station conservation
    assert np.all(inflow <= CAP * s + tol)                  # capacity, only open stations
    assert np.all(u[:, x == 0] < tol) and np.all(w[:, x == 0] < tol)
    assert np.isclose(u.sum() + w.sum(), q.sum())           # everything ends at a landfill
    assert np.isclose(r["transport"] + r["stations_cost"], r["obj"])

def report(r):
    inflow = r["v"].sum(axis=0)
    print("Landfills built:        ", np.flatnonzero(r["x"]) + 1)
    print("Transfer stations built:", np.flatnonzero(r["s"]) + 1)
    print(f"  # stations = {int(r['s'].sum())}, tons through stations = {inflow.sum():,.1f} "
          f"({inflow.sum() / (r['u'].sum() + inflow.sum()):.1%} of waste), max station load = {inflow.max():,.1f}")
    print(f"Distance traveled by waste: {r['miles']:,.1f} mile-tons/day")
    print(f"Transportation cost:        ${r['transport']:,.2f}/day")
    print(f"Transfer station cost:      ${r['stations_cost']:,.2f}/day")
    print(f"Total daily cost:           ${r['obj']:,.2f}/day")

r1 = solve_network(centers, landfills, stations, q, 5)
check(r1, q, 5)
report(r1)

Landfills built:         [ 3  4  7 13 14]
Transfer stations built: [ 4 12 21 23 30 31 32 39 45]
  # stations = 9, tons through stations = 17,137.5 (35.1% of waste), max station load = 2,000.0
Distance traveled by waste: 876,496.8 mile-tons/day
Transportation cost:        $718,642.75/day
Transfer station cost:      $90,000.00/day
Total daily cost:           $808,642.75/day


The optimal solution builds landfills 3, 4, 7, 13, and 14 and transfer stations 4, 12, 21, 23, 30, 31, 32, 39, and 45. The total distance traveled is 876,496.8 mile-tons per day, with a transportation cost of $718,642.75 and transfer station operating costs of $90,000, giving a total daily cost of \$808,642.75.

**Results (region 1).**

| | |
|---|---|
| Landfills built | 3, 4, 7, 13, 14 |
| Transfer stations built | 4, 12, 21, 23, 30, 31, 32, 39, 45 (9 stations) |
| Distance traveled by waste | 876,496.8 mile-tons/day |
| Transportation cost | \$718,642.75/day |
| Transfer station cost | \$90,000/day |
| **Total daily cost** | **\$808,642.75/day** |

The landfills are the same as in (b). About 35% of the waste (17,137.5 t/day) now goes through stations. The total mile-tons actually goes up a bit compared to (b) (876k vs 840k) because of the detours, but because the second leg is at half price, the total cost goes down from \$840,488 to \$808,643 per day, even after paying for the stations.

## Part C. A second region

### (e) Region 2 on its own

Same model as (d), just with the region 2 data.

In [4]:
r2 = solve_network(centers2, landfills2, stations2, q2, 5)
check(r2, q2, 5)
report(r2)

Landfills built:         [ 6  8  9 13 15]
Transfer stations built: [ 6  7 24 31 32 36]
  # stations = 6, tons through stations = 10,900.2 (13.9% of waste), max station load = 2,000.0
Distance traveled by waste: 1,246,638.4 mile-tons/day
Transportation cost:        $1,091,761.23/day
Transfer station cost:      $60,000.00/day
Total daily cost:           $1,151,761.23/day


For Region 2, the optimal solution builds landfills 6, 8, 9, 13, and 15 and transfer stations 6, 7, 24, 31, 32, and 36. The total weighted distance is 1,246,638.4 mile-tons per day, with transportation costs of 1,091,761.23 and transfer station operating costs of 60,000, giving a total daily cost of 1,151,761.23.

**Results (region 2).**

| | |
|---|---|
| Landfills built | 6, 8, 9, 13, 15 |
| Transfer stations built | 6, 7, 24, 31, 32, 36 (6 stations) |
| Distance traveled by waste | 1,246,638.4 mile-tons/day |
| Transportation cost | \$1,091,761.23/day |
| Transfer station cost | \$60,000/day |
| **Total daily cost** | **\$1,151,761.23/day** |

Region 2 makes more waste than region 1 (78.6k vs 48.8k t/day) but uses fewer stations: only about 14% of its waste goes through a station.

### (f) Joint planning across both regions

I stack the two data sets (90 centers, 30 candidate landfills, 100 candidate stations), require $\sum_j x_j = 10$, and allow any flow between the regions since the problem doesn't restrict it. Landfill indices 1–15 and station indices 1–50 are region 1, the rest are region 2 (shown as R1-#/R2-# using the original numbering).

In [5]:
C_all = np.vstack([centers, centers2])
L_all = np.vstack([landfills, landfills2])
S_all = np.vstack([stations, stations2])
q_all = np.concatenate([q, q2])

rj = solve_network(C_all, L_all, S_all, q_all, 10)
check(rj, q_all, 10)

def label(idx, n1):
    return [f"R1-{i + 1}" if i < n1 else f"R2-{i - n1 + 1}" for i in idx]

nL1, nS1, nC1 = len(landfills), len(stations), len(centers)
print("Landfills built:        ", label(np.flatnonzero(rj["x"]), nL1))
print("Transfer stations built:", label(np.flatnonzero(rj["s"]), nS1))
print(f"  # stations = {int(rj['s'].sum())}, tons through stations = {rj['v'].sum():,.1f}")
print(f"Distance traveled by waste: {rj['miles']:,.1f} mile-tons/day")
print(f"Transportation cost:        ${rj['transport']:,.2f}/day")
print(f"Transfer station cost:      ${rj['stations_cost']:,.2f}/day")
print(f"Total daily cost:           ${rj['obj']:,.2f}/day")

# waste disposed in each region vs. waste generated there
disposed = rj["u"].sum(axis=0) + rj["w"].sum(axis=0)
print(f"\nRegion 1: generates {q.sum():,.1f} t/day, disposes {disposed[:nL1].sum():,.1f} t/day")
print(f"Region 2: generates {q2.sum():,.1f} t/day, disposes {disposed[nL1:].sum():,.1f} t/day")

Landfills built:         ['R1-2', 'R1-3', 'R1-14', 'R2-1', 'R2-2', 'R2-6', 'R2-8', 'R2-9', 'R2-13', 'R2-15']
Transfer stations built: ['R1-1', 'R1-4', 'R1-6', 'R1-11', 'R1-12', 'R1-30', 'R1-32', 'R1-39', 'R1-45', 'R1-46', 'R1-48', 'R2-7', 'R2-24', 'R2-27', 'R2-32']
  # stations = 15, tons through stations = 27,701.8
Distance traveled by waste: 2,008,649.7 mile-tons/day
Transportation cost:        $1,715,384.50/day
Transfer station cost:      $150,000.00/day
Total daily cost:           $1,865,384.50/day

Region 1: generates 48,803.2 t/day, disposes 38,419.0 t/day
Region 2: generates 78,624.9 t/day, disposes 89,009.1 t/day


In [6]:
separate = r1["obj"] + r2["obj"]
savings = separate - rj["obj"]

pd.DataFrame({
    "Separate (d)+(e)": [r1["x"].sum() + r2["x"].sum(), r1["s"].sum() + r2["s"].sum(), r1["miles"] + r2["miles"],
                         r1["transport"] + r2["transport"], r1["stations_cost"] + r2["stations_cost"], separate],
    "Joint (f)": [rj["x"].sum(), rj["s"].sum(), rj["miles"], rj["transport"], rj["stations_cost"], rj["obj"]],
}, index=["Landfills", "Transfer stations", "Mile-tons/day", "Transport cost ($/day)",
          "Station cost ($/day)", "Total cost ($/day)"]).assign(Difference=lambda df: df["Joint (f)"] - df["Separate (d)+(e)"]).round(0)

,Separate (d)+(e),Joint (f),Difference
Landfills,10.0,10.0,0.0
Transfer stations,15.0,15.0,0.0
Mile-tons/day,2123135.0,2008650.0,-114485.0
Transport cost ($/day),1810404.0,1715385.0,-95019.0
Station cost ($/day),150000.0,150000.0,0.0
Total cost ($/day),1960404.0,1865385.0,-95019.0


In [7]:
print(f"Daily savings from joint planning: ${savings:,.2f} ({savings / separate:.1%})")
print(f"Annual savings (365 days):         ${365 * savings:,.0f}")

Daily savings from joint planning: $95,019.48 (4.8%)
Annual savings (365 days):         $34,682,111


**Results.** The joint plan builds landfills R1-2, R1-3, R1-14 and R2-1, R2-2, R2-6, R2-8, R2-9, R2-13, R2-15 (3 in region 1, 7 in region 2), plus 15 transfer stations (11 in region 1, 4 in region 2). The **total daily cost is 1,865,384.50**: 1,715,384.50 for transportation and 150,000 for stations. Waste travels 2,008,649.7 mile-tons/day. Planning the regions separately costs 808,642.75 + 1,151,761.23 = 1,960,403.98 per day, so the joint plan saves **95,019.48/day (4.8%)**, or about 34.7M per year. Both plans build the same number of facilities, so all of the savings come from transportation.

I recommend planning the waste management system **jointly** across the two regions. The joint solution reduces total daily cost from 1,960,403.98 to 1,865,384.50, saving approximately 95,019 per day, or 4.8\%, while using the same total number of landfills and transfer stations. The savings come entirely from lower transportation costs because joint planning allows waste to cross the regional boundary and use more efficient facilities. However, the resulting system is unevenly distributed, with Region 1 building only 3 of the 10 landfills and sending about 10,400 tons of waste per day to Region 2. Therefore, the two regions should agree on a cost-sharing arrangement before implementing the joint plan.